<a href="https://colab.research.google.com/github/Cav1on/ml-playground/blob/main/05_pytorch_basics_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Week 6: PyTorch Basics
Objective: Understand and implement a basic neural network training loop in PyTorch.
Tasks:
1. Prepare data in PyTorch frmat(Dataset and DataLoader).
2. Create a simple neural network(nn.Module)
3. Write a training loop: forward -> loss -> backward -> step.

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Download and clean the data (as in week 4)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df = df.drop(['Ticket', 'Cabin', 'Name', 'PassengerId'], axis = 1)
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = pd.get_dummies(df, columns = ['Sex', 'Embarked'], drop_first = True)

scaler = StandardScaler()
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])

x = df.drop('Survived', axis = 1)
y = df['Survived']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)
print('Data for PyTorch is ready!')

Data for PyTorch is ready!


In [12]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Create your own Dataset class
class TitanicDataset(Dataset):
  def __init__(self, x, y):
    # Converting Pandas tables into PyThourc tensors(matrices)
    self.x = torch.tensor(x.values.astype('float32'), dtype = torch.float32)
    # 'y' needs to be converted to column [N , 1] so that the dimensions match
    self.y = torch.tensor(y.values.astype('float32'), dtype = torch.float32).unsqueeze(1)

  def __len__(self):
    # Returns the total number of rows
    return len(self.x)

  def __getitem__(self, idx):
    # Returns one pair(features, answer)
    return self.x[idx], self.y[idx]

# 2. Initializing Dataset
train_dataset = TitanicDataset(x_train, y_train)

# 3. Create a DataLoader(it will output data in chunks of 32 pieces)
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)

print(f"Number of bathce(portions) in DataLoader: {len(train_loader)}")

Number of bathce(portions) in DataLoader: 23


In [13]:
import torch.nn as nn

class SimpleMLP(nn.Module):
  def __init__(self, input_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(input_features, 16), # Hidden layer: we create 16 neurons from the input features
        nn.ReLU(), # Activation function (adds nonlinearity)
        nn.Linear(16, 1), # Output layer: from 16 neurons we make 1 answer
        nn.Sigmoid() # Compresses the answer into a range between 0 and 1 (survival probability)
    )

  def forward(self, x):
  # How data flows through the network
    return self.network(x)

# Create a model (pass the number of columns to x_train)
model = SimpleMLP(input_features = x_train.shape[1])
print(model)

SimpleMLP(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
    (3): Sigmoid()
  )
)


In [16]:
import torch.optim as optim

# 1. Loss function - estimates how much the model is wrong
criterion = nn.BCELoss() # Binary Cross Entropy - the standard for yes/no problems

# 2. Optimizer - updates the model weights to reduce the error
optimizer = optim.Adam(model.parameters(), lr = 0.01)

epochs = 20 # Number of passes through all data

print('Let s start learning...\n')
for epoch in range(epochs):
  # Let's go through all the batches from DataLoader
  for x_bathc, y_batch in train_loader:

    # Step 1: Prediction (Forward Pass)
    preds = model(x_bathc)

    # Step 2: Calculate the error
    loss = criterion(preds, y_batch)

    # Step 3: Reset Old Gradients
    optimizer.zero_grad()

    # Step 4: Calculate new gradients (Backward pass)
    loss.backward()

    # Step 5: Updating Weights (Optimizer Step)
    optimizer.step()

  # We print an error at the end of every 5th epoch
  if(epoch + 1) % 5 == 0:
    print(f"{epoch + 1} / {epochs} | Error(Loss): {loss.item():.4f}")

print("\nThe training is complete!")

Let s start learning...

5 / 20 | Error(Loss): 0.3071
10 / 20 | Error(Loss): 0.2791
15 / 20 | Error(Loss): 0.1305
20 / 20 | Error(Loss): 0.2422

The training is complete!
